In [ ]:
import numpy as np
import pandas as pd
from datasets import Dataset, load_dataset

# =======================================================
# 1. 3-Bin Stratified Stream Sampling (is_read == True)
# =======================================================
# Target: 50,000 total rows distributed evenly across 3 duration bins
BIN_TARGETS = {
    "short": 16667,   # <= 14 days
    "medium": 16667,  # 15 to 90 days
    "long": 16666     # > 90 days
}


def get_duration_bin(days):
    """Classify duration into one of the 3 target brackets."""
    if days <= 14:
        return "short"
    elif days <= 90:
        return "medium"
    else:
        return "long"


print("Step 1: Loading 'behavior' dataset in streaming mode...")
behavior_stream = load_dataset("liyucheng/goodreads", "behavior", streaming=True)

processed_splits = {}

for split_name, dataset_iterable in behavior_stream.items():
    print(f"\n--- Processing split: {split_name} ---")

    bin_counts = {"short": 0, "medium": 0, "long": 0}
    selected_rows = []

    for example in dataset_iterable:
        is_read = example.get("is_read")
        duration = example.get("reading_duration_days")

        # Condition 1: Must be read
        # Condition 2: Must have a valid duration > 0 days
        if is_read is True and duration is not None and not pd.isna(duration) and duration > 0:
            category = get_duration_bin(duration)

            if bin_counts[category] < BIN_TARGETS[category]:
                bin_counts[category] += 1
                example["duration_bin"] = category
                selected_rows.append(example)

        # Terminate streaming once all 3 bins reach their quota
        if all(bin_counts[b] >= BIN_TARGETS[b] for b in BIN_TARGETS):
            print(f"Target of 50,000 rows reached for split '{split_name}'.")
            break

    print(f"Collected bin distribution for '{split_name}': {bin_counts}")

    if selected_rows:
        df_split = pd.DataFrame(selected_rows)
        # Log-transformed duration to compress the long tail for ML tasks
        df_split["log_reading_duration"] = np.log1p(df_split["reading_duration_days"])
        processed_splits[split_name] = df_split
    else:
        print(f"Warning: No valid records collected for '{split_name}'.")

# =======================================================
# 2. Book Metadata Extraction & Author ID Resolution
# =======================================================
print("\nStep 2: Loading 'books' dataset...")
books_ds = load_dataset("liyucheng/goodreads", "books")
books_data = books_ds["train"] if "train" in books_ds else next(iter(books_ds.values()))
books_df = books_data.to_pandas()


def extract_primary_author(authors_val):
    """Extract the first/primary author_id from author dict or list."""
    if isinstance(authors_val, (list, np.ndarray)) and len(authors_val) > 0:
        first = authors_val[0]
        if isinstance(first, dict):
            return str(first.get("author_id", ""))
        return str(first)
    elif isinstance(authors_val, dict):
        return str(authors_val.get("author_id", ""))
    elif pd.notna(authors_val):
        return str(authors_val)
    return ""


# Extract author_id if present in nested format
if "authors" in books_df.columns:
    books_df["author_id"] = books_df["authors"].apply(extract_primary_author)
elif "author_id" in books_df.columns:
    books_df["author_id"] = books_df["author_id"].astype(str)

desired_book_cols = [
    "book_id", "title", "author_id", "num_pages",
    "publication_year", "genres", "description", "average_rating"
]
available_book_cols = [c for c in desired_book_cols if c in books_df.columns]
books_clean = books_df[available_book_cols].drop_duplicates(subset=["book_id"])
books_clean["book_id"] = books_clean["book_id"].astype(str)

# =======================================================
# 3. Author Name Extraction & Resolution
# =======================================================
print("\nStep 3: Loading 'authors' dataset to fetch author names...")
try:
    authors_ds = load_dataset("liyucheng/goodreads", "authors")
    authors_data = authors_ds["train"] if "train" in authors_ds else next(iter(authors_ds.values()))
    authors_df = authors_data.to_pandas()

    # Identify the author ID and Name columns dynamically
    id_col = next((c for c in ["author_id", "id"] if c in authors_df.columns), None)
    name_col = next((c for c in ["name", "author_name"] if c in authors_df.columns), None)

    if id_col and name_col:
        authors_df[id_col] = authors_df[id_col].astype(str)
        author_lookup = (
            authors_df[[id_col, name_col]]
            .rename(columns={id_col: "author_id", name_col: "author_name"})
            .drop_duplicates(subset=["author_id"])
        )
        # Merge author names directly into books catalog
        books_clean = pd.merge(books_clean, author_lookup, on="author_id", how="left")
        print("Successfully merged author names with book metadata.")
    else:
        print("Warning: Could not identify ID/Name columns in 'authors' dataset.")
        books_clean["author_name"] = None
except Exception as e:
    print(f"Notice: Could not load 'authors' subset ({e}). Falling back without authors dataset.")
    if "author_name" not in books_clean.columns:
        books_clean["author_name"] = None

# =======================================================
# 4. Final Merge and Export
# =======================================================
print("\nStep 4: Merging all data splits...")

for split_name, df_split in processed_splits.items():
    df_split["book_id"] = df_split["book_id"].astype(str)

    print(f"Merging {split_name} interactions with book & author metadata...")
    merged_df = pd.merge(df_split, books_clean, on="book_id", how="left")

    output_file = f"balanced_read_50k_{split_name}.csv"
    merged_df.to_csv(output_file, index=False)

    print(f"\nCreated file: {output_file}")
    print(f"Total rows: {len(merged_df)}")
    print(f"is_read distribution (all True):\n{merged_df['is_read'].value_counts()}")
    print(f"Duration bin distribution:\n{merged_df['duration_bin'].value_counts()}")
    print(f"Author name non-null count: {merged_df['author_name'].notna().sum()} / {len(merged_df)}\n")

print("Pipeline complete.")